# 📚 Book Recommendation System
### Semantic Similarity via `all-MiniLM-L6-v2` + FAISS Vector Store

**Pipeline:**
1. Load dataset (40 books across 10 categories)
2. Build embedding text from metadata + description
3. Encode with `sentence-transformers/all-MiniLM-L6-v2`
4. Index embeddings into a FAISS flat L2 index
5. Query: given a book title → find top-K similar books

---

## 2. Imports & Config

In [1]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
TOP_K      = 5
CSV_PATH   = 'books.csv'

print(f'Model  : {MODEL_NAME}')
print(f'Top-K  : {TOP_K}')
print(f'Dataset: {CSV_PATH}')

c:\Users\qusaikald\Desktop\demo-book-recoo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model  : sentence-transformers/all-MiniLM-L6-v2
Top-K  : 5
Dataset: books.csv


## 3. Load Dataset

In [ ]:
df = pd.read_csv(CSV_PATH)

print(f'Books loaded : {len(df)}')
print(f'Categories   : {sorted(df.category.unique())}')
print(f'\nCategory distribution:')
print(df.category.value_counts().to_string())
print()
df.head(5)

## 4. Build Embedding Text

Each book is represented as a single string that combines structured metadata with its description. This gives the model both categorical signals and semantic content.

**Format:** `[Category] <Title> by <Author> (<Year>) | <Description>`

In [ ]:
def build_embedding_text(row: pd.Series) -> str:
    """Combine metadata + description into a single string for embedding."""
    return (
        f"[{row['category']}] {row['title']} by {row['author']} ({row['year']}) | "
        f"{row['description']}"
    )

df['embedding_text'] = df.apply(build_embedding_text, axis=1)

# Preview
print('Sample embedding text:\n')
for i in range(3):
    print(f'  [{i}] {df.embedding_text.iloc[i][:120]}...')
    print()

## 5. Load Model & Encode

In [4]:
print(f'Loading model: {MODEL_NAME} ...')
model = SentenceTransformer(MODEL_NAME, device='cpu')

print(f'  Embedding dim : {model.get_sentence_embedding_dimension()}')
print(f'  Max seq len   : {model.max_seq_length}')

print(f'\nEncoding {len(df)} books ...')
embeddings = model.encode(
    df['embedding_text'].tolist(),
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = embeddings.astype(np.float32)
print(f'\nEmbedding matrix shape: {embeddings.shape}')

Loading model: sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2066.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding dim : 384
  Max seq len   : 256

Encoding 40 books ...


Batches: 100%|██████████| 5/5 [00:02<00:00,  1.75it/s]


Embedding matrix shape: (40, 384)


In [5]:
embeddings[0]

array([-4.47179563e-02,  5.43663464e-02, -6.60627708e-02, -3.59779298e-02,
        6.63660420e-03, -6.10332377e-02,  1.16026536e-01,  6.87575638e-02,
       -1.06508404e-01,  2.39278991e-02, -6.80939155e-03,  3.40665691e-02,
        8.17141086e-02, -8.31324682e-02,  3.02973967e-02,  6.74123317e-02,
       -1.36987105e-01, -4.83832583e-02,  8.74268338e-02, -4.43707556e-02,
       -1.01695042e-02, -1.40566882e-02, -6.81687891e-02,  1.21084461e-02,
       -1.98000036e-02,  3.80392373e-02,  7.46009275e-02, -1.01907030e-01,
        7.26733310e-03,  2.42660125e-03, -7.67930374e-02,  6.52272180e-02,
        8.87660892e-04,  6.42658174e-02,  1.53943617e-02,  7.22506046e-02,
       -2.57191230e-02,  3.82579342e-02, -5.13529964e-02,  9.01623862e-04,
       -1.02666624e-01,  1.11600822e-02, -3.47378366e-02, -3.24193053e-02,
       -2.28885282e-02, -5.80890216e-02, -3.56020294e-02, -3.27938795e-02,
       -8.21954608e-02, -1.89927872e-02, -1.23091839e-01, -6.54184967e-02,
        5.74104674e-02, -

## 6. Build FAISS Index

Since embeddings are L2-normalized, an **Inner Product (IP)** index is equivalent to cosine similarity search — fast, no approximation needed for 40 books.

In [6]:
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # Inner Product (= cosine sim after L2 norm)
index.add(embeddings)

print(f'FAISS index type   : IndexFlatIP (exact, cosine similarity)')
print(f'Embedding dimension: {dim}')
print(f'Vectors indexed    : {index.ntotal}')

FAISS index type   : IndexFlatIP (exact, cosine similarity)
Embedding dimension: 384
Vectors indexed    : 40


## 7. Recommendation Function

In [7]:
def get_similar_books(title: str, top_k: int = TOP_K, verbose: bool = True) -> pd.DataFrame:
    """
    Given a book title, return the top_k most similar books.

    Args:
        title   : Exact or partial title (case-insensitive match)
        top_k   : Number of recommendations to return
        verbose : Print formatted table to stdout

    Returns:
        DataFrame of recommended books with similarity scores
    """
    # --- resolve title ---
    mask    = df['title'].str.lower().str.contains(title.lower())
    matches = df[mask]
    if matches.empty:
        raise ValueError(f'No book found matching: "{title}"')

    query_row = matches.iloc[0]
    query_idx = query_row.name

    # --- embed query ---
    query_vec = embeddings[query_idx].reshape(1, -1)   # reuse cached embedding

    # --- FAISS search (top_k + 1 to exclude the query itself) ---
    scores, indices = index.search(query_vec, top_k + 1)
    scores, indices = scores[0], indices[0]

    # --- filter out query book ---
    results = [
        {**df.iloc[idx].to_dict(), 'similarity': float(score)}
        for idx, score in zip(indices, scores)
        if idx != query_idx
    ][:top_k]


    

    result_df = pd.DataFrame(results)

    if verbose:
        print(f'\n🔍 Query book')
        print(f'   Title    : {query_row["title"]}')
        print(f'   Author   : {query_row["author"]}')
        print(f'   Category : {query_row["category"]}')
        print(f'   Rating   : {query_row["rating"]}')
        print(f'   Desc     : {query_row["description"][:100]}...')

        print(f'\n📚 Top-{top_k} Similar Books\n')
        display_cols = ['title', 'author', 'category', 'year', 'rating', 'similarity']
        display_df   = result_df[display_cols].copy()
        display_df['similarity'] = display_df['similarity'].map('{:.4f}'.format)
        print(tabulate(display_df, headers='keys', tablefmt='rounded_outline', showindex=False))

    return result_df

print('Function `get_similar_books` ready.')

Function `get_similar_books` ready.


## 8. Demo Queries

### Query 1 — Sci-fi to Sci-fi

In [ ]:
_ = get_similar_books('Dune')

### Query 2 — Fantasy to Fantasy

In [16]:
_ = get_similar_books('The Hobbit' )


🔍 Query book
   Title    : The Hobbit
   Author   : J.R.R. Tolkien
   Category : Fantasy
   Rating   : 4.8
   Desc     : The classic fantasy adventure of Bilbo Baggins, a homebody hobbit who is swept into an epic quest to...

📚 Top-5 Similar Books

╭──────────────────────────────────────┬────────────────────┬─────────────────┬────────┬──────────┬──────────────╮
│ title                                │ author             │ category        │   year │   rating │   similarity │
├──────────────────────────────────────┼────────────────────┼─────────────────┼────────┼──────────┼──────────────┤
│ The Way of Kings                     │ Brandon Sanderson  │ Fantasy         │   2010 │      4.8 │       0.3905 │
│ The Name of the Wind                 │ Patrick Rothfuss   │ Fantasy         │   2007 │      4.8 │       0.3891 │
│ A Game of Thrones                    │ George R.R. Martin │ Fantasy         │   1996 │      4.7 │       0.3348 │
│ The Hitchhiker's Guide to the Galaxy │ Douglas Adams      

### Query 3 — Tech/Engineering

In [17]:
_ = get_similar_books('Clean Code' , top_k=2)


🔍 Query book
   Title    : Clean Code
   Author   : Robert C. Martin
   Category : Technology
   Rating   : 4.7
   Desc     : A handbook of agile software craftsmanship that teaches programmers how to write readable, maintaina...

📚 Top-2 Similar Books

╭──────────────────────────┬──────────────┬────────────┬────────┬──────────┬──────────────╮
│ title                    │ author       │ category   │   year │   rating │   similarity │
├──────────────────────────┼──────────────┼────────────┼────────┼──────────┼──────────────┤
│ The Pragmatic Programmer │ David Thomas │ Technology │   1999 │      4.8 │       0.6235 │
│ The Lean Startup         │ Eric Ries    │ Business   │   2011 │      4.5 │       0.35   │
╰──────────────────────────┴──────────────┴────────────┴────────┴──────────┴──────────────╯


### Query 4 — Psychology/Philosophy cross-domain

In [ ]:
_ = get_similar_books("Man's Search for Meaning")

### Query 5 — Business

In [13]:
_ = get_similar_books('Zero to One')


🔍 Query book
   Title    : Zero to One
   Author   : Peter Thiel
   Category : Business
   Rating   : 4.5
   Desc     : Notes on startups and how to build the future, arguing that true innovation means creating something...

📚 Top-5 Similar Books

╭────────────────────────────────────┬─────────────────┬─────────────┬────────┬──────────┬──────────────╮
│ title                              │ author          │ category    │   year │   rating │   similarity │
├────────────────────────────────────┼─────────────────┼─────────────┼────────┼──────────┼──────────────┤
│ The Lean Startup                   │ Eric Ries       │ Business    │   2011 │      4.5 │       0.4948 │
│ Good to Great                      │ Jim Collins     │ Business    │   2001 │      4.6 │       0.3614 │
│ The Subtle Art of Not Giving a Fck │ Mark Manson     │ Self-Help   │   2016 │      4.5 │       0.3406 │
│ Atomic Habits                      │ James Clear     │ Non-Fiction │   2018 │      4.8 │       0.3332 │
│ The Fou

## 9. Similarity Score Distribution

Inspect how similarity scores are distributed for a given query — useful for tuning a relevance threshold.

In [11]:
import matplotlib.pyplot as plt

def plot_similarity_distribution(title: str, top_k: int = 15):
    mask      = df['title'].str.lower().str.contains(title.lower())
    query_idx = df[mask].index[0]
    query_vec = embeddings[query_idx].reshape(1, -1)

    scores, indices = index.search(query_vec, top_k + 1)
    scores  = scores[0]
    indices = indices[0]

    # filter self
    pairs = [(df.iloc[i]['title'], s) for i, s in zip(indices, scores) if i != query_idx][:top_k]
    titles_list, scores_list = zip(*pairs)

    colors = ['#4f86c6' if s >= 0.7 else '#f4a259' if s >= 0.5 else '#e06c75' for s in scores_list]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(range(len(titles_list)), scores_list, color=colors, edgecolor='none', height=0.65)
    ax.set_yticks(range(len(titles_list)))
    ax.set_yticklabels([t[:45] for t in titles_list], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Cosine Similarity', fontsize=10)
    ax.set_title(f'Similarity scores for: "{df.iloc[query_idx]["title"]}"', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.axvline(0.7, color='#4f86c6', linestyle='--', linewidth=0.8, label='≥0.70 high')
    ax.axvline(0.5, color='#f4a259', linestyle='--', linewidth=0.8, label='≥0.50 mid')
    ax.legend(fontsize=8, loc='lower right')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_similarity_distribution('Dune', top_k=15)

ModuleNotFoundError: No module named 'matplotlib'

## 10. Category Confusion Matrix

For each book, check what category its top-1 nearest neighbor belongs to. Measures how well the embedding space clusters genres.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

categories = sorted(df['category'].unique())
cat2idx    = {c: i for i, c in enumerate(categories)}
n_cats     = len(categories)
matrix     = np.zeros((n_cats, n_cats), dtype=int)

# For each book, find top-1 neighbour (excl. self)
for i, row in df.iterrows():
    q_vec = embeddings[i].reshape(1, -1)
    scores, idxs = index.search(q_vec, 2)  # top-2: skip self
    neighbor_idx = idxs[0][1] if idxs[0][0] == i else idxs[0][0]
    true_cat   = cat2idx[row['category']]
    pred_cat   = cat2idx[df.iloc[neighbor_idx]['category']]
    matrix[true_cat][pred_cat] += 1

fig, ax = plt.subplots(figsize=(10, 8))
cmap = mcolors.LinearSegmentedColormap.from_list('', ['#1a1a2e', '#4f86c6', '#a8dadc'])
im   = ax.imshow(matrix, cmap=cmap, aspect='auto')

ax.set_xticks(range(n_cats))
ax.set_yticks(range(n_cats))
ax.set_xticklabels(categories, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(categories, fontsize=9)
ax.set_xlabel('Nearest Neighbour Category', fontsize=10)
ax.set_ylabel('Query Book Category', fontsize=10)
ax.set_title('Category Coherence — Top-1 Nearest Neighbour', fontsize=12, fontweight='bold')

for i in range(n_cats):
    for j in range(n_cats):
        if matrix[i, j] > 0:
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center',
                    fontsize=11, color='white' if matrix[i,j] < 3 else 'black', fontweight='bold')

plt.colorbar(im, ax=ax, label='Count')
plt.tight_layout()
plt.show()

# Diagonal accuracy
same_cat = int(np.diag(matrix).sum())
print(f'\nTop-1 same-category rate: {same_cat}/{len(df)} = {same_cat/len(df):.1%}')

## 11. Save & Reload FAISS Index

Demonstrates FAISS persistence — encode once, reload for inference.

In [12]:
import os

INDEX_PATH = 'books.faiss'
faiss.write_index(index, INDEX_PATH)
print(f'Index saved to: {INDEX_PATH}  ({os.path.getsize(INDEX_PATH):,} bytes)')

# Reload and verify
index_loaded = faiss.read_index(INDEX_PATH)
print(f'Reloaded index vectors: {index_loaded.ntotal}')

# Sanity check — same results?
q = embeddings[0].reshape(1, -1)
s1, i1 = index.search(q, 3)
s2, i2 = index_loaded.search(q, 3)
assert np.allclose(s1, s2), 'Mismatch after reload!'
print('Reload verification: ✓ identical results')

Index saved to: books.faiss  (61,485 bytes)
Reloaded index vectors: 40
Reload verification: ✓ identical results


---
## Summary

| Component | Choice | Notes |
|-----------|--------|-------|
| Embedding Model | `all-MiniLM-L6-v2` | 384-dim, fast, strong semantic quality |
| Vector Store | FAISS `IndexFlatIP` | Exact cosine sim, no approximation |
| Dataset | 40 books, 10 categories | Metadata + description fused into embedding text |
| Normalization | L2-normalize | Makes IP index equivalent to cosine similarity |
| Persistence | `faiss.write_index` / `read_index` | Encode once, reload freely |

**Scaling path:**
- Swap `IndexFlatIP` → `IndexIVFFlat` or `IndexHNSWFlat` for millions of vectors
- Replace local FAISS with ChromaDB / Qdrant for multi-process / multi-tenant access
- Upgrade model to `all-mpnet-base-v2` or a domain-tuned variant for better quality